1. Load df_match_features
2. Train baseline check if needed
3. Train no-conf model
4. Train no-conf + Elo cap model
5. Save experimental models
6. Quick validation comparison

# Experiment A — Remove confederation features

In [3]:
from pathlib import Path
import pickle

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [4]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
EXPERIMENT_MODELS_DIR = MODELS_DIR / "experiments" / "no_conf"

EXPERIMENT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Experiment model output:", EXPERIMENT_MODELS_DIR)

Project root: /Users/riadanas/Desktop/Football WC2026
Experiment model output: /Users/riadanas/Desktop/Football WC2026/models/experiments/no_conf


In [5]:
# df = pd.read_csv(PROCESSED_DIR / "df_match_features.csv")

df = pd.read_csv(PROCESSED_DIR / "df_match_features.csv")

print(df.shape)
display(df.head())
display(df.columns.tolist())

(49048, 16)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0.000000,1,False,UEFA,UEFA
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1502.801300,1497.198700,5.602600,1,False,UEFA,UEFA
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1486.622531,1513.377469,-26.754938,1,False,UEFA,UEFA
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1505.454946,1494.545054,10.909891,1,False,UEFA,UEFA
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1497.633108,1502.366892,-4.733783,1,False,UEFA,UEFA


['date',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'tournament',
 'city',
 'country',
 'neutral',
 'home_elo_pre',
 'away_elo_pre',
 'elo_diff',
 'tournament_weight',
 'is_competitive',
 'home_confederation',
 'away_confederation']

In [6]:
required_columns = [
    "home_score",
    "away_score",
    "home_elo_pre",
    "away_elo_pre",
    "elo_diff",
    "tournament_weight",
    "neutral",
]

missing_columns = [col for col in required_columns if col not in df.columns]

print("Missing columns:", missing_columns)

Missing columns: []


In [ ]:
### Build no-conf feature set
### Only use numerical features

feature_cols_no_conf = [
    "home_elo_pre",
    "away_elo_pre",
    "elo_diff",
    "tournament_weight",
    "neutral",
]

target_home = "home_score"
target_away = "away_score"

df_model = df.dropna(
    subset=feature_cols_no_conf + [target_home, target_away]
).copy()

X = df_model[feature_cols_no_conf].copy()
X = X.astype(float)

X = sm.add_constant(X, has_constant="add")

y_home = df_model[target_home].astype(float)
y_away = df_model[target_away].astype(float)

print("X shape:", X.shape)
print("Home target shape:", y_home.shape)
print("Away target shape:", y_away.shape)

display(X.head())

X shape: (49048, 6)
Home target shape: (49048,)
Away target shape: (49048,)


,const,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,neutral
0,1.0,1500.000000,1500.000000,0.000000,1.0,0.0
1,1.0,1502.801300,1497.198700,5.602600,1.0,0.0
2,1.0,1486.622531,1513.377469,-26.754938,1.0,0.0
3,1.0,1505.454946,1494.545054,10.909891,1.0,0.0
4,1.0,1497.633108,1502.366892,-4.733783,1.0,0.0


In [8]:
X.columns.tolist()

['const',
 'home_elo_pre',
 'away_elo_pre',
 'elo_diff',
 'tournament_weight',
 'neutral']

In [9]:
#### Train no-conf Poisson models

model_home_no_conf = sm.GLM(
    y_home,
    X,
    family=sm.families.Poisson()
).fit()

model_away_no_conf = sm.GLM(
    y_away,
    X,
    family=sm.families.Poisson()
).fit()

print(model_home_no_conf.summary())
print(model_away_no_conf.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             home_score   No. Observations:                49048
Model:                            GLM   Df Residuals:                    49043
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -82058.
Date:                Thu, 04 Jun 2026   Deviance:                       68057.
Time:                        15:33:25   Pearson chi2:                 6.77e+04
No. Iterations:                   100   Pseudo R-squ. (CS):             0.2674
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 1.2433      0.03

In [10]:
### Quick validation comparison

home_pred = model_home_no_conf.predict(X)
away_pred = model_away_no_conf.predict(X)

home_mae = mean_absolute_error(y_home, home_pred)
away_mae = mean_absolute_error(y_away, away_pred)

home_rmse = mean_squared_error(y_home, home_pred) ** 0.5
away_rmse = mean_squared_error(y_away, away_pred) ** 0.5

print("No-conf model validation")
print("-" * 30)
print(f"Home goals MAE: {home_mae:.3f}")
print(f"Away goals MAE: {away_mae:.3f}")
print(f"Home goals RMSE: {home_rmse:.3f}")
print(f"Away goals RMSE: {away_rmse:.3f}")

print("\nActual vs predicted means")
print(f"Actual home goals mean: {y_home.mean():.3f}")
print(f"Pred home goals mean:   {home_pred.mean():.3f}")
print(f"Actual away goals mean: {y_away.mean():.3f}")
print(f"Pred away goals mean:   {away_pred.mean():.3f}")

No-conf model validation
------------------------------
Home goals MAE: 1.148
Away goals MAE: 0.917
Home goals RMSE: 1.603
Away goals RMSE: 1.278

Actual vs predicted means
Actual home goals mean: 1.755
Pred home goals mean:   1.755
Actual away goals mean: 1.180
Pred away goals mean:   1.180


In [11]:
### Save experiment model files

with open(EXPERIMENT_MODELS_DIR / "poisson_home.pkl", "wb") as f:
    pickle.dump(model_home_no_conf, f)

with open(EXPERIMENT_MODELS_DIR / "poisson_away.pkl", "wb") as f:
    pickle.dump(model_away_no_conf, f)

with open(EXPERIMENT_MODELS_DIR / "feature_columns.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)

print("Saved no-conf experiment models to:")
print(EXPERIMENT_MODELS_DIR)

Saved no-conf experiment models to:
/Users/riadanas/Desktop/Football WC2026/models/experiments/no_conf


In [12]:
#### Sanity check reload

with open(EXPERIMENT_MODELS_DIR / "poisson_home.pkl", "rb") as f:
    loaded_home = pickle.load(f)

with open(EXPERIMENT_MODELS_DIR / "poisson_away.pkl", "rb") as f:
    loaded_away = pickle.load(f)

with open(EXPERIMENT_MODELS_DIR / "feature_columns.pkl", "rb") as f:
    loaded_feature_columns = pickle.load(f)

print(loaded_feature_columns)

test_pred_home = loaded_home.predict(X.head(3))
test_pred_away = loaded_away.predict(X.head(3))

print("Test home predictions:", test_pred_home.tolist())
print("Test away predictions:", test_pred_away.tolist())

['const', 'home_elo_pre', 'away_elo_pre', 'elo_diff', 'tournament_weight', 'neutral']
Test home predictions: [1.71389405046484, 1.7309232783818003, 1.6348537808068768]
Test away predictions: [1.0940478553549142, 1.0824811442938151, 1.1510115928348295]


#
----

# Experiment A2 — No confederation, clean Elo features

- I removed elo_diff compared to experimetn A

In [13]:
EXPERIMENT_MODELS_DIR_CLEAN = MODELS_DIR / "experiments" / "no_conf_clean"
EXPERIMENT_MODELS_DIR_CLEAN.mkdir(parents=True, exist_ok=True)

feature_cols_no_conf_clean = [
    "home_elo_pre",
    "away_elo_pre",
    "tournament_weight",
    "neutral",
]

target_home = "home_score"
target_away = "away_score"

df_model_clean = df.dropna(
    subset=feature_cols_no_conf_clean + [target_home, target_away]
).copy()

X_clean = df_model_clean[feature_cols_no_conf_clean].copy()
X_clean = X_clean.astype(float)
X_clean = sm.add_constant(X_clean, has_constant="add")

y_home_clean = df_model_clean[target_home].astype(float)
y_away_clean = df_model_clean[target_away].astype(float)

print("X_clean shape:", X_clean.shape)
display(X_clean.head())
print(X_clean.columns.tolist())

X_clean shape: (49048, 5)


,const,home_elo_pre,away_elo_pre,tournament_weight,neutral
0,1.0,1500.000000,1500.000000,1.0,0.0
1,1.0,1502.801300,1497.198700,1.0,0.0
2,1.0,1486.622531,1513.377469,1.0,0.0
3,1.0,1505.454946,1494.545054,1.0,0.0
4,1.0,1497.633108,1502.366892,1.0,0.0


['const', 'home_elo_pre', 'away_elo_pre', 'tournament_weight', 'neutral']


In [14]:
model_home_no_conf_clean = sm.GLM(
    y_home_clean,
    X_clean,
    family=sm.families.Poisson()
).fit()

model_away_no_conf_clean = sm.GLM(
    y_away_clean,
    X_clean,
    family=sm.families.Poisson()
).fit()

print(model_home_no_conf_clean.summary())
print(model_away_no_conf_clean.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             home_score   No. Observations:                49048
Model:                            GLM   Df Residuals:                    49043
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -82058.
Date:                Thu, 04 Jun 2026   Deviance:                       68057.
Time:                        15:40:29   Pearson chi2:                 6.77e+04
No. Iterations:                     5   Pseudo R-squ. (CS):             0.2674
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 1.2434      0.03

In [15]:
home_pred_clean = model_home_no_conf_clean.predict(X_clean)
away_pred_clean = model_away_no_conf_clean.predict(X_clean)

home_mae_clean = mean_absolute_error(y_home_clean, home_pred_clean)
away_mae_clean = mean_absolute_error(y_away_clean, away_pred_clean)

home_rmse_clean = mean_squared_error(y_home_clean, home_pred_clean) ** 0.5
away_rmse_clean = mean_squared_error(y_away_clean, away_pred_clean) ** 0.5

print("No-conf clean model validation")
print("-" * 35)
print(f"Home goals MAE: {home_mae_clean:.3f}")
print(f"Away goals MAE: {away_mae_clean:.3f}")
print(f"Home goals RMSE: {home_rmse_clean:.3f}")
print(f"Away goals RMSE: {away_rmse_clean:.3f}")

print("\nActual vs predicted means")
print(f"Actual home goals mean: {y_home_clean.mean():.3f}")
print(f"Pred home goals mean:   {home_pred_clean.mean():.3f}")
print(f"Actual away goals mean: {y_away_clean.mean():.3f}")
print(f"Pred away goals mean:   {away_pred_clean.mean():.3f}")

No-conf clean model validation
-----------------------------------
Home goals MAE: 1.148
Away goals MAE: 0.917
Home goals RMSE: 1.603
Away goals RMSE: 1.278

Actual vs predicted means
Actual home goals mean: 1.755
Pred home goals mean:   1.755
Actual away goals mean: 1.180
Pred away goals mean:   1.180


#### this tells us: We can remove confederation features without breaking the model.

- We have not yet confirmed whether this fixes the South America bias.

Why?

- Because that bias appears in the tournament simulation output, not only in model training metrics.

In [16]:
with open(EXPERIMENT_MODELS_DIR_CLEAN / "poisson_home.pkl", "wb") as f:
    pickle.dump(model_home_no_conf_clean, f)

with open(EXPERIMENT_MODELS_DIR_CLEAN / "poisson_away.pkl", "wb") as f:
    pickle.dump(model_away_no_conf_clean, f)

with open(EXPERIMENT_MODELS_DIR_CLEAN / "feature_columns.pkl", "wb") as f:
    pickle.dump(X_clean.columns.tolist(), f)

print("Saved no-conf clean experiment models to:")
print(EXPERIMENT_MODELS_DIR_CLEAN)

Saved no-conf clean experiment models to:
/Users/riadanas/Desktop/Football WC2026/models/experiments/no_conf_clean


#
----

# Experiment B — Cap Elo difference